## Thư viện

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import warnings
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix, precision_recall_fscore_support

warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', None)

In [ ]:
base_path = '../../data/hwu/'

try:
    df_train = pd.read_csv(os.path.join(base_path, 'train.csv'), sep=',')
    df_val = pd.read_csv(os.path.join(base_path, 'val.csv'), sep=',')
    df_test = pd.read_csv(os.path.join(base_path, 'test.csv'), sep=',')

    df_train.rename(columns={'category': 'intent'}, inplace=True)
    df_val.rename(columns={'category': 'intent'}, inplace=True)
    df_test.rename(columns={'category': 'intent'}, inplace=True)

except FileNotFoundError:
    print(f"Lỗi: Không tìm thấy file. Đã kiểm tra đường dẫn: {os.path.abspath(base_path)}")
    print("Hãy chắc chắn cấu trúc thư mục của bạn là: project_root/data/hwu/[các file .csv]")

print("Train shape:", df_train.shape)
print("Validation shape:", df_val.shape)
print("Test shape:", df_test.shape)

print("\n--- 5 mẫu dữ liệu training: ---")
display(df_train.head())

# 2. Tiền xử lý Nhãn (LabelEncoder)
print("\n--- Đang mã hóa nhãn... ---")

# Thêm .dropna() để loại bỏ bất kỳ giá trị NaN nào có thể sót lại
all_intents = pd.concat([df_train['intent'], df_val['intent'], df_test['intent']]).dropna().unique()
le = LabelEncoder()
le.fit(all_intents)

# Transform các tập (cũng thêm .dropna() cho an toàn, dù không cần thiết nếu file sạch)
y_train = le.transform(df_train['intent'].dropna())
y_val = le.transform(df_val['intent'].dropna())
y_test = le.transform(df_test['intent'].dropna())

num_classes = len(le.classes_)
print(f"Tổng số lớp (intent): {num_classes}") # <-- Sẽ ra số lớn (ví dụ: 68)
print(f"Ví dụ mã hóa: '{df_train['intent'].iloc[0]}' -> {y_train[0]}")
print(f"Số lượng nhãn duy nhất trong y_train: {np.unique(y_train).size}") # <-- Sẽ ra số lớn

# Tạo một dictionary để lưu kết quả
results_summary = {}

Train shape: (8954, 2)
Validation shape: (1076, 2)
Test shape: (1076, 2)

--- 5 mẫu dữ liệu training: ---


,text,intent
0,what alarms do i have set right now,alarm_query
1,checkout today alarm of meeting,alarm_query
2,report alarm settings,alarm_query
3,see see for me the alarms that you have set tomorrow morning,alarm_query
4,is there an alarm for ten am,alarm_query



--- Đang mã hóa nhãn... ---
Tổng số lớp (intent): 64
Ví dụ mã hóa: 'alarm_query' -> 0
Số lượng nhãn duy nhất trong y_train: 64


## Nhiệm vụ 1: (Warm-up) Pipeline TF-IDF + Logistic Regression

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

print("--- Đang huấn luyện Mô hình 1: TF-IDF + Logistic Regression ---")

# 1. Tạo pipeline
tfidf_lr_pipeline = make_pipeline(
    TfidfVectorizer(max_features=5000),
    LogisticRegression(max_iter=1000, random_state=42)
)

# 2. Huấn luyện pipeline trên tập train
# Dùng df_train['text'] và y_train (đã được tạo đúng ở Bước 0)
tfidf_lr_pipeline.fit(df_train['text'], y_train)

# 3. Đánh giá trên tập test
y_pred_tfidf = tfidf_lr_pipeline.predict(df_test['text'])

print("\n--- Kết quả Mô hình 1 (trên tập Test) ---")
print(classification_report(y_test, y_pred_tfidf, target_names=le.classes_, zero_division=0))

# Lưu kết quả
f1_macro_tfidf = f1_score(y_test, y_pred_tfidf, average='macro', zero_division=0)
results_summary['TF-IDF + LR'] = {'f1_macro': f1_macro_tfidf, 'test_loss': 'N/A'}

--- Đang huấn luyện Mô hình 1: TF-IDF + Logistic Regression ---

--- Kết quả Mô hình 1 (trên tập Test) ---
                          precision    recall  f1-score   support

             alarm_query       0.90      0.95      0.92        19
            alarm_remove       1.00      0.73      0.84        11
               alarm_set       0.77      0.89      0.83        19
       audio_volume_down       1.00      0.75      0.86         8
       audio_volume_mute       0.92      0.80      0.86        15
         audio_volume_up       0.93      1.00      0.96        13
          calendar_query       0.45      0.53      0.49        19
         calendar_remove       0.89      0.89      0.89        19
            calendar_set       0.87      0.68      0.76        19
          cooking_recipe       0.59      0.68      0.63        19
        datetime_convert       0.67      0.75      0.71         8
          datetime_query       0.74      0.89      0.81        19
        email_addcontact       0.7

## Nhiệm vụ 2: (Warm-up Ôn bài cũ) Pipeline Word2Vec (Trung bình) + Dense Layer

In [8]:
!pip install gensim

from gensim.models import Word2Vec
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
# -------------------------------------

print("--- Đang chuẩn bị Mô hình 2: Word2Vec (Average) + Dense ---")

# 1. Huấn luyện mô hình Word2Vec (chỉ trên tập train)
sentences = [str(text).split() for text in df_train['text']]
embedding_dim_w2v = 100 # Chọn chiều vector là 100

w2v_model = Word2Vec(sentences, vector_size=embedding_dim_w2v, window=5, min_count=1, workers=4)
print(f"Đã huấn luyện Word2Vec (vector_size={embedding_dim_w2v})")

# 2. Viết hàm chuyển câu thành vector trung bình
def sentence_to_avg_vector(text, model):
    words = str(text).split()
    word_vectors = [model.wv[word] for word in words if word in model.wv]

    if not word_vectors:
        return np.zeros(model.vector_size)

    return np.mean(word_vectors, axis=0)

# 3. Tạo dữ liệu train/val/test
X_train_avg = np.array([sentence_to_avg_vector(text, w2v_model) for text in df_train['text']])
X_val_avg = np.array([sentence_to_avg_vector(text, w2v_model) for text in df_val['text']])
X_test_avg = np.array([sentence_to_avg_vector(text, w2v_model) for text in df_test['text']])

# 4. Xây dựng mô hình Sequential của Keras
model_avg = Sequential([
    Dense(128, activation='relu', input_shape=(embedding_dim_w2v,)),
    Dropout(0.5),
    Dense(num_classes, activation='softmax') # Output layer
])

model_avg.summary()

# 5. Compile, huấn luyện
model_avg.compile(
    loss='sparse_categorical_crossentropy', # Dùng sparse vì y là số nguyên
    optimizer='adam',
    metrics=['accuracy']
)

early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("\n--- Đang huấn luyện Mô hình 2 ---")
history_avg = model_avg.fit(
    X_train_avg, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_val_avg, y_val),
    callbacks=[early_stopping],
    verbose=1
)

# 6. Đánh giá
print("\n--- Kết quả Mô hình 2 (trên tập Test) ---")
test_loss_avg, test_acc_avg = model_avg.evaluate(X_test_avg, y_test)
y_pred_avg_probs = model_avg.predict(X_test_avg)
y_pred_avg = np.argmax(y_pred_avg_probs, axis=1)

print(classification_report(y_test, y_pred_avg, target_names=le.classes_, zero_division=0))

# Lưu kết quả
f1_macro_avg = f1_score(y_test, y_pred_avg, average='macro', zero_division=0)
results_summary['Word2Vec (Avg) + Dense'] = {'f1_macro': f1_macro_avg, 'test_loss': test_loss_avg}

--- Đang chuẩn bị Mô hình 2: Word2Vec (Average) + Dense ---
Đã huấn luyện Word2Vec (vector_size=100)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 128)            │        12,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,184 (82.75 KB)

 Trainable params: 21,184 (82.75 KB)

 Non-trainable params: 0 (0.00 B)


--- Đang huấn luyện Mô hình 2 ---
Epoch 1/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.0206 - loss: 4.1551 - val_accuracy: 0.0316 - val_loss: 4.1055
Epoch 2/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.0356 - loss: 4.1059 - val_accuracy: 0.0548 - val_loss: 4.0425
Epoch 3/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0471 - loss: 4.0331 - val_accuracy: 0.0623 - val_loss: 3.9286
Epoch 4/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0632 - loss: 3.9270 - val_accuracy: 0.0939 - val_loss: 3.8024
Epoch 5/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0684 - loss: 3.8181 - val_accuracy: 0.0985 - val_loss: 3.6925
Epoch 6/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0825 - loss: 3.7140 - val_accuracy: 0.0939 - val_loss: 3.6052
Epoch 7/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0953 - loss: 3.6385 - val_accuracy: 0.1301 - val_loss: 3.5353
Epoch 8/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.099

## Nhiệm vụ 3 & 4

In [10]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Các tham số
vocab_size = 10000     # Giới hạn 10000 từ vựng (bao gồm cả <UNK>)
max_len = 50           # Chiều dài tối đa của một câu

# 2. Tokenizer: Tạo vocab và chuyển text thành chuỗi chỉ số
# Đảm bảo text là string
train_texts = df_train['text'].astype(str)
val_texts = df_val['text'].astype(str)
test_texts = df_test['text'].astype(str)

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<UNK>")
tokenizer.fit_on_texts(train_texts) # Chỉ fit trên tập train

word_index = tokenizer.word_index
print(f"Kích thước từ vựng (tất cả từ): {len(word_index)}")

# 3. Chuyển text thành chuỗi (sequences)
train_sequences = tokenizer.texts_to_sequences(train_texts)
val_sequences = tokenizer.texts_to_sequences(val_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)

# 4. Padding
X_train_pad = pad_sequences(train_sequences, maxlen=max_len, padding='post', truncating='post')
X_val_pad = pad_sequences(val_sequences, maxlen=max_len, padding='post', truncating='post')
X_test_pad = pad_sequences(test_sequences, maxlen=max_len, padding='post', truncating='post')

print(f"\nShape của X_train_pad: {X_train_pad.shape}")
print(f"Ví dụ một câu sau khi xử lý (câu đầu tiên):")
print(f"Text gốc: {train_texts.iloc[0]}")
print(f"Padded:   {X_train_pad[0]}")

Kích thước từ vựng (tất cả từ): 4264

Shape của X_train_pad: (8954, 50)
Ví dụ một câu sau khi xử lý (câu đầu tiên):
Text gốc: what alarms do i have set right now
Padded:   [ 9 99 24  5 26 35 92 62  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0]


## Nhiệm vụ 3: Mô hình Nâng cao (Embedding Pre-trained + LSTM)

In [12]:
from tensorflow.keras.layers import Embedding, SpatialDropout1D, LSTM
from tensorflow.keras.models import Sequential

# 1. Tạo ma trận trọng số (Embedding Matrix)
embedding_matrix = np.zeros((vocab_size, embedding_dim_w2v))

for word, i in word_index.items():
    if i >= vocab_size:
        continue
    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]

print(f"Shape của Ma trận Embedding: {embedding_matrix.shape}")

# 2. Xây dựng mô hình
lstm_model_pretrained = Sequential([
    Embedding(
        input_dim=vocab_size,         # 10000
        output_dim=embedding_dim_w2v, # 100
        weights=[embedding_matrix],   # Tải trọng số pre-trained
        input_length=max_len,
        trainable=False               # Đóng băng lớp Embedding
    ),
    SpatialDropout1D(0.2),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(num_classes, activation='softmax')
])

lstm_model_pretrained.summary()

# 3. Compile và Huấn luyện
lstm_model_pretrained.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

print("\n--- Đang huấn luyện Mô hình 3 (Pre-trained LSTM) ---")
history_pretrained = lstm_model_pretrained.fit(
    X_train_pad, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_val_pad, y_val),
    callbacks=[early_stopping],
    verbose=1
)

# 4. Đánh giá
print("\n--- Kết quả Mô hình 3 (trên tập Test) ---")
test_loss_pre, test_acc_pre = lstm_model_pretrained.evaluate(X_test_pad, y_test)
y_pred_pre_probs = lstm_model_pretrained.predict(X_test_pad)
y_pred_pre = np.argmax(y_pred_pre_probs, axis=1)

print(classification_report(y_test, y_pred_pre, target_names=le.classes_, zero_division=0))

# Lưu kết quả
f1_macro_pre = f1_score(y_test, y_pred_pre, average='macro', zero_division=0)
results_summary['LSTM (Pre-trained)'] = {'f1_macro': f1_macro_pre, 'test_loss': test_loss_pre}

Shape của Ma trận Embedding: (10000, 100)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,000,000 (3.81 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 1,000,000 (3.81 MB)


--- Đang huấn luyện Mô hình 3 (Pre-trained LSTM) ---
Epoch 1/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 53s 173ms/step - accuracy: 0.0180 - loss: 4.1496 - val_accuracy: 0.0260 - val_loss: 4.0634
Epoch 2/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 72s 138ms/step - accuracy: 0.0301 - loss: 4.0683 - val_accuracy: 0.0502 - val_loss: 3.9179
Epoch 3/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 39s 132ms/step - accuracy: 0.0430 - loss: 3.9442 - val_accuracy: 0.0520 - val_loss: 3.9130
Epoch 4/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 41s 146ms/step - accuracy: 0.0453 - loss: 3.9246 - val_accuracy: 0.0669 - val_loss: 3.7719
Epoch 5/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 35s 126ms/step - accuracy: 0.0437 - loss: 3.8445 - val_accuracy: 0.0651 - val_loss: 3.7265
Epoch 6/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 37s 133ms/step - accuracy: 0.0544 - loss: 3.8138 - val_accuracy: 0.0743 - val_loss: 3.7183
Epoch 7/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 39s 138ms/step - accuracy: 0.0587 - loss: 3.7859 - val_accuracy: 0.0929 - val_loss: 3.6512
Epoch 8/20
280/280 ━━━━━━━━━━

## Nhiệm vụ 4: Mô hình Nâng cao (Embedding học từ đầu + LSTM)

In [13]:
embedding_dim_scratch = 100 # Chọn 100 chiều

# 1. Xây dựng mô hình
lstm_model_scratch = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim_scratch,
        input_length=max_len
    ),
    SpatialDropout1D(0.2),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(num_classes, activation='softmax')
])

lstm_model_scratch.summary()

# 2. Compile và Huấn luyện
lstm_model_scratch.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

print("\n Đang huấn luyện Mô hình 4 (LSTM học từ đầu) ")
history_scratch = lstm_model_scratch.fit(
    X_train_pad, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_val_pad, y_val),
    callbacks=[early_stopping],
    verbose=1
)

# 3. Đánh giá
print("\n--- Kết quả Mô hình 4 (trên tập Test) ---")
test_loss_scratch, test_acc_scratch = lstm_model_scratch.evaluate(X_test_pad, y_test)
y_pred_scratch_probs = lstm_model_scratch.predict(X_test_pad)
y_pred_scratch = np.argmax(y_pred_scratch_probs, axis=1)

print(classification_report(y_test, y_pred_scratch, target_names=le.classes_, zero_division=0))

# Lưu kết quả
f1_macro_scratch = f1_score(y_test, y_pred_scratch, average='macro', zero_division=0)
results_summary['LSTM (Scratch)'] = {'f1_macro': f1_macro_scratch, 'test_loss': test_loss_scratch}

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_1             │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


 Đang huấn luyện Mô hình 4 (LSTM học từ đầu) 
Epoch 1/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 66s 165ms/step - accuracy: 0.0168 - loss: 4.1499 - val_accuracy: 0.0177 - val_loss: 4.1281
Epoch 2/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 41s 145ms/step - accuracy: 0.0165 - loss: 4.1347 - val_accuracy: 0.0177 - val_loss: 4.1246
Epoch 3/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 42s 150ms/step - accuracy: 0.0146 - loss: 4.1330 - val_accuracy: 0.0177 - val_loss: 4.1247
Epoch 4/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 82s 149ms/step - accuracy: 0.0134 - loss: 4.1324 - val_accuracy: 0.0177 - val_loss: 4.1250
Epoch 5/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 82s 150ms/step - accuracy: 0.0142 - loss: 4.1326 - val_accuracy: 0.0177 - val_loss: 4.1251

--- Kết quả Mô hình 4 (trên tập Test) ---
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.0288 - loss: 4.1579
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step
                          precision    recall  f1-score   support

             alarm_query       0.00      0.00      0.00        19
        

## Nhiệm vụ 5: Đánh giá, So sánh và Phân tích

In [14]:
df_results = pd.DataFrame.from_dict(results_summary, orient='index', columns=['f1_macro', 'test_loss'])
df_results.index.name = 'Pipeline'
df_results = df_results.reset_index()

print("--- Bảng tổng hợp kết quả trên tập Test ---")
display(df_results.sort_values(by='f1_macro', ascending=False))

--- Bảng tổng hợp kết quả trên tập Test ---


,Pipeline,f1_macro,test_loss
0,TF-IDF + LR,0.835298,N/A
1,Word2Vec (Avg) + Dense,0.135866,3.140813
2,LSTM (Pre-trained),0.058210,3.454363
3,LSTM (Scratch),0.000542,4.124564


In [15]:
def predict_intent(text, model, model_type):
    text = str(text) # Đảm bảo là string
    # Model 1: TF-IDF
    if model_type == 'tfidf':
        pred = model.predict([text])
        return le.inverse_transform(pred)[0]

    # Model 2: Word2Vec Avg
    if model_type == 'w2v_avg':
        avg_vec = np.array([sentence_to_avg_vector(text, w2v_model)])
        pred_probs = model.predict(avg_vec)
        pred = np.argmax(pred_probs, axis=1)
        return le.inverse_transform(pred)[0]

    # Model 3 & 4: LSTM
    if model_type == 'lstm':
        seq = tokenizer.texts_to_sequences([text])
        pad = pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')
        pred_probs = model.predict(pad)
        pred = np.argmax(pred_probs, axis=1)
        return le.inverse_transform(pred)[0]

# Các câu test khó
test_examples = [
    ("can you remind me to not call my mom", "reminder_create"),
    ("is it going to be sunny or rainy tomorrow", "weather_query"),
    ("find a flight from new york to london but not through paris", "flight_search")
]

# Tạo bảng kết quả
analysis_data = []
for text, true_intent in test_examples:
    pred_tfidf = predict_intent(text, tfidf_lr_pipeline, 'tfidf')
    pred_w2v = predict_intent(text, model_avg, 'w2v_avg')
    pred_lstm_pre = predict_intent(text, lstm_model_pretrained, 'lstm')
    pred_lstm_scratch = predict_intent(text, lstm_model_scratch, 'lstm')

    analysis_data.append({
        "Câu test": text,
        "Nhãn đúng": true_intent,
        "Dự đoán TF-IDF": pred_tfidf,
        "Dự đoán W2V (Avg)": pred_w2v,
        "Dự đoán LSTM (Pre)": pred_lstm_pre,
        "Dự đoán LSTM (Scratch)": pred_lstm_scratch
    })

df_analysis = pd.DataFrame(analysis_data)
display(df_analysis)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step


,Câu test,Nhãn đúng,Dự đoán TF-IDF,Dự đoán W2V (Avg),Dự đoán LSTM (Pre),Dự đoán LSTM (Scratch)
0,can you remind me to not call my mom,reminder_create,calendar_set,general_explain,datetime_query,email_query
1,is it going to be sunny or rainy tomorrow,weather_query,weather_query,calendar_query,qa_currency,email_query
2,find a flight from new york to london but not through paris,flight_search,general_negate,takeaway_order,email_sendemail,email_query
